In [9]:
#the optimal neural network hyperparameters

In [10]:
import sys
sys.path.insert(1, "C:/Users/hp/Downloads/RL-X/one_policy_to_run_them_all/sentence_transformer")

import optuna

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import csv
from torch.utils.data import Dataset, DataLoader, TensorDataset
from semanticreasoning import ProcessData


In [11]:
#load tainset

In [12]:
with open("../commands.csv", "r") as f:
    reader = csv.reader(f)

    next(reader)

    commands = []
    texts = []
    for row in reader:
        if not any(row):
            continue
        text = row[0]
        texts.append(text)
        command = [float(x) for x in row[1:4]]
        commands.append(command)


['move forward', '1', '0', '0', 'forward']
['go forward', '1', '0', '0', 'forward']
['walk forward', '1', '0', '0', 'forward']
['move ahead', '1', '0', '0', 'forward']
['go straight', '1', '0', '0', 'forward']
['continue forward', '1', '0', '0', 'forward']
['advance', '1', '0', '0', 'forward']
['head forward', '1', '0', '0', 'forward']
['proceed ahead', '1', '0', '0', 'forward']
['move onward', '1', '0', '0', 'forward']
['move backward', '-1', '0', '0', 'backward']
['go backward', '-1', '0', '0', 'backward']
['walk back', '-1', '0', '0', 'backward']
['reverse', '-1', '0', '0', 'backward']
['move in reverse', '-1', '0', '0', 'backward']
['step backward', '-1', '0', '0', 'backward']
['go back', '-1', '0', '0', 'backward']
['retreat', '-1', '0', '0', 'backward']
['back up', '-1', '0', '0', 'backward']
['move back', '-1', '0', '0', 'backward']
['strafe left', '0', '-1', '0', 'strafe_left']
['move left', '0', '-1', '0', 'strafe_left']
['shift left', '0', '-1', '0', 'strafe_left']
['slide le

In [13]:
transformer = ProcessData()

embeddings = transformer.getEmbeddings(texts)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1819.23it/s]


In [14]:
embeddings = torch.tensor(embeddings, dtype=torch.float32)
commands = torch.tensor(commands, dtype=torch.float32)
train_dataset = TensorDataset(embeddings, commands)
train_loader = DataLoader(train_dataset, batch_size=20, shuffle=True)

C:\Users\hp\AppData\Local\Temp\ipykernel_23072\3212805297.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  embeddings = torch.tensor(embeddings, dtype=torch.float32)


In [15]:
#loading test set

with open("../validation_commands.csv", "r") as f:
    reader = csv.reader(f)

    next(reader)

    commands = []
    texts = []
    for row in reader:
        if not any(row):
            continue
        text = row[0]
        texts.append(text)
        command = [float(x) for x in row[1:4]]
        commands.append(command)


In [16]:
embeddings = transformer.getEmbeddings(texts)

In [17]:
embeddings = torch.tensor(embeddings, dtype=torch.float32)
commands = torch.tensor(commands, dtype=torch.float32)
valid_dataset = TensorDataset(embeddings, commands)
valid_loader = DataLoader(train_dataset, batch_size=20, shuffle=True)

C:\Users\hp\AppData\Local\Temp\ipykernel_23072\3333564261.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  embeddings = torch.tensor(embeddings, dtype=torch.float32)


In [18]:
OUTPUT = 3
INPUT_SIZE = 384
DEVICE = torch.device("cpu")
EPOCHS = 1000

In [19]:
def define_model(trial):
    n_layers = trial.suggest_int("n_layers", 1, 3)
    layers = []

    in_features = INPUT_SIZE
    for i in range(n_layers):
        out_features = trial.suggest_int("n_units{}".format(i), 4, 128)
        layers.append(nn.Linear(in_features, out_features))
        layers.append(nn.ReLU())
        p = trial.suggest_float("dropout_l{}".format(i), 0.2, 0.5)
        layers.append(nn.Dropout(p))

        in_features = out_features

    layers.append(nn.Linear(in_features, OUTPUT))

    return nn.Sequential(*layers)

In [20]:
def objective(trial):
    #generate the model
    model = define_model(trial).to(DEVICE)

    #generate the optimizers
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop", "SGD"])
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    optimizer = getattr(optim, optimizer_name)(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    #train model
    for epoch in range(EPOCHS):
        #training
        model.train()
        for data, target in train_loader:
            
            data, target = data.float().to(DEVICE), target.float().to(DEVICE)

            optimizer.zero_grad()
            predictions = model(data)
            loss = loss_fn(predictions, target)
            loss.backward()
            optimizer.step()

        #validation 
        model.eval()

        total_squared_error = 0.0
        total_elements = 0
        with torch.no_grad():
            for data, target in valid_loader:
                data, target = data.float().to(DEVICE), target.float().to(DEVICE)
                predictions = model(data)

                total_squared_error += (
                    (predictions-target).pow(2).sum().item()
                )

                total_elements += target.numel()

        validation_mse = total_squared_error / total_elements

        trial.report(validation_mse, epoch)

        #Handle pruning based on the intermediate value
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
        
    return validation_mse

In [21]:
study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner()
)

study.optimize(objective, n_trials=100)

print("Best validation MSE:", study.best_value)
print("Best parameters:", study.best_params)

[I 2026-07-23 22:51:37,384] A new study created in memory with name: no-name-d17e2888-5214-43b7-a8f3-9cb4799eeae8
[I 2026-07-23 22:52:20,839] Trial 0 finished with value: 0.020305111010869344 and parameters: {'n_layers': 3, 'n_units0': 8, 'dropout_l0': 0.2624276744068196, 'n_units1': 73, 'dropout_l1': 0.35673538636747765, 'n_units2': 65, 'dropout_l2': 0.29340794326773767, 'optimizer': 'Adam', 'lr': 0.0021502429486227343}. Best is trial 0 with value: 0.020305111010869344.
[I 2026-07-23 22:52:54,035] Trial 1 finished with value: 0.08913470831784336 and parameters: {'n_layers': 3, 'n_units0': 5, 'dropout_l0': 0.3708370310118395, 'n_units1': 93, 'dropout_l1': 0.42610954433072634, 'n_units2': 32, 'dropout_l2': 0.3837676826765549, 'optimizer': 'Adam', 'lr': 0.0006239532607821614}. Best is trial 0 with value: 0.020305111010869344.
[I 2026-07-23 22:53:24,762] Trial 2 finished with value: 0.004886934477271456 and parameters: {'n_layers': 2, 'n_units0': 43, 'dropout_l0': 0.34090344505317793, 'n_

Best validation MSE: 0.0012277572895541336
Best parameters: {'n_layers': 1, 'n_units0': 74, 'dropout_l0': 0.2235646078578695, 'optimizer': 'Adam', 'lr': 0.004767710151759911}
